# Evaluación de Naive Bayes (Scikit-Learn)

Este notebook selecciona la mejor configuración por **F1 macro medio anual** mediante
`RandomizedSearchCV` o `GridSearchCV` para un clasificador **`GaussianNB`** de scikit-learn,
utilizando **los mismos folds temporales** que el modelo de árbol de decisión manual.

Toda la validación cruzada utiliza exclusivamente `train` (partidos anteriores al 1 de enero de 2024).
El conjunto `test` queda reservado y no se utiliza en esta etapa.

In [ ]:
# 1. Carga del dataset y definición de atributos
import pandas as pd
from load import load_dataset
from pipeline import pipeline_input_attributes

dataset = load_dataset("futbol_uruguayo.csv")
print("Primeras filas del dataset:")
print(dataset.head())

In [ ]:
# 2. Separación temporal entre desarrollo (train) y test
test_start = pd.Timestamp("2024-01-01")

train = dataset[dataset["date"] < test_start].copy()
test = dataset[dataset["date"] >= test_start].copy()

assert not train.empty, "El conjunto de entrenamiento está vacío"
assert (train["date"] < test_start).all()
assert (test["date"] >= test_start).all()

print("Partidos para entrenamiento y validación cruzada:", len(train))
print("Partidos de test reservados desde 2024 (sin evaluar):", len(test))

X_train = train[pipeline_input_attributes].copy()
y_train = train["result"].copy()

In [ ]:
# 3. Construcción de los folds temporales idénticos
import numpy as np

validation_years = list(range(2013, 2023))
train_years = train["date"].dt.year.to_numpy()
window_years = 5
temporal_splits = []

for validation_year in validation_years:
    fit_indices = np.flatnonzero(
        (train_years < validation_year) & (train_years >= validation_year - window_years)
    )
    validation_indices = np.flatnonzero(
        train_years == validation_year
    )

    assert len(fit_indices) > 0 and len(validation_indices) > 0, "Fold vacío"
    assert train.iloc[fit_indices]["date"].max() < train.iloc[validation_indices]["date"].min()
    assert (train.iloc[validation_indices]["date"] < test_start).all()

    temporal_splits.append((fit_indices, validation_indices))
    print(f"Validación {validation_year}: entrenamiento={len(fit_indices)}, validación={len(validation_indices)}")

In [ ]:
# 4. Búsqueda de hiperparámetros con Naive Bayes (GaussianNB)
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV

nb = GaussianNB()

# Se ajusta 'var_smoothing' para estabilizar las varianzas de las variables continuas
param_grid = {
    "var_smoothing": np.logspace(0, -9, num=100)
}

selection_metric = "f1_macro"
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
}

search_nb = GridSearchCV(
    estimator=nb,
    param_grid=param_grid,
    scoring=scoring,
    cv=temporal_splits,
    n_jobs=-1,
    refit=selection_metric,
    verbose=1,
)

search_nb.fit(X_train, y_train)

print("\nHiperparámetros seleccionados por F1 macro medio anual (Naive Bayes):")
print(search_nb.best_params_)
print(f"F1 macro medio anual del ganador: {search_nb.best_score_:.3%}")
best_acc = search_nb.cv_results_["mean_test_accuracy"][search_nb.best_index_]
print(f"Accuracy media anual del mismo ganador: {best_acc:.3%}")
best_balanced_acc = search_nb.cv_results_["mean_test_balanced_accuracy"][search_nb.best_index_]
print(f"Balanced accuracy media anual del mismo ganador: {best_balanced_acc:.3%}")

In [ ]:
# 5. Tabla de candidatos ordenada por F1 macro medio anual
from IPython.display import display

results_nb = pd.DataFrame(search_nb.cv_results_)
parameter_columns = [col for col in results_nb.columns if col.startswith("param_")]
metric_names = [search_nb.refit] + [m for m in scoring if m != search_nb.refit]
summary_columns = [
    f"{stat}_test_{metric}"
    for metric in metric_names
    for stat in ("mean", "std", "rank")
]

results_nb = results_nb[parameter_columns + summary_columns].sort_values(
    f"rank_test_{search_nb.refit}", kind="stable"
)

print("Primeros 10 candidatos de Naive Bayes (ordenados por F1 macro):")
display(results_nb.head(10))

In [ ]:
# 6. Matrices de confusión y reporte agrupado de validación
from sklearn.base import clone
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
import matplotlib.pyplot as plt

real_results = []
predicted_results = []

for fit_indices, validation_indices in temporal_splits:
    fold_model = clone(search_nb.best_estimator_)
    fold_model.fit(X_train.iloc[fit_indices], y_train.iloc[fit_indices])
    fold_predictions = fold_model.predict(X_train.iloc[validation_indices])
    
    real_results.extend(y_train.iloc[validation_indices])
    predicted_results.extend(fold_predictions)

real_results = np.asarray(real_results)
predicted_results = np.asarray(predicted_results)
labels = ["L", "E", "V"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    real_results,
    predicted_results,
    labels=labels,
    display_labels=["Local", "Empate", "Visitante"],
    cmap="Blues",
    values_format="d",
    ax=axes[0],
)
axes[0].set_title("Naive Bayes - Cantidades agrupadas")

ConfusionMatrixDisplay.from_predictions(
    real_results,
    predicted_results,
    labels=labels,
    display_labels=["Local", "Empate", "Visitante"],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    ax=axes[1],
)
axes[1].set_title("Naive Bayes - Proporciones por clase real")

plt.tight_layout()
plt.show()

metrics_summary = pd.DataFrame({
    "Media anual (CV)": {
        "F1 macro": search_nb.cv_results_["mean_test_f1_macro"][search_nb.best_index_],
        "Accuracy": search_nb.cv_results_["mean_test_accuracy"][search_nb.best_index_],
        "Balanced accuracy": search_nb.cv_results_["mean_test_balanced_accuracy"][search_nb.best_index_],
    },
    "Validación agrupada": {
        "F1 macro": f1_score(real_results, predicted_results, average="macro", zero_division=0),
        "Accuracy": accuracy_score(real_results, predicted_results),
        "Balanced accuracy": balanced_accuracy_score(real_results, predicted_results),
    },
})
print("Métricas resumidas de Naive Bayes:")
print(metrics_summary.to_string(float_format=lambda v: f"{v:.3%}"))
print("\nReporte por clase (Validación agrupada):")
print(classification_report(
    real_results, predicted_results, labels=labels,
    target_names=["Local", "Empate", "Visitante"], zero_division=0
))

In [ ]:
# 7. Registro del experimento en un CSV propio (experimentos_nb.csv)
import csv
import json
from datetime import datetime, timezone
from pathlib import Path
import load

csv_path = Path(load.__file__).resolve().parent / "experimentos_nb.csv"
best_index = search_nb.best_index_

experiment = {
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "modelo": "GaussianNB",
    "validation_years": json.dumps(validation_years),
    "window_years": window_years,
    "n_train": len(X_train),
    "n_test_reservado": len(test),
    "mejores_parametros": json.dumps(search_nb.best_params_, sort_keys=True),
    "refit": search_nb.refit,
    "cv_accuracy": search_nb.cv_results_["mean_test_accuracy"][best_index],
    "cv_f1_macro": search_nb.cv_results_["mean_test_f1_macro"][best_index],
    "cv_balanced_accuracy": search_nb.cv_results_["mean_test_balanced_accuracy"][best_index],
    "reporte_validacion": classification_report(
        real_results, predicted_results, labels=labels,
        target_names=["Local", "Empate", "Visitante"], zero_division=0,
    ),
    "notas": "GaussianNB sobre variables continuas; test sin evaluar",
}

experiment["reporte_validacion"] = " | ".join(
    " ".join(line.split())
    for line in experiment["reporte_validacion"].splitlines() if line.strip()
)

has_header = csv_path.exists() and csv_path.stat().st_size > 0
with csv_path.open("a", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=experiment)
    if not has_header:
        writer.writeheader()
    writer.writerow(experiment)

print(f"Experimento guardado exitosamente en {csv_path}")